# M2-CL on PACS — reproduction pipeline (Colab, T4)

This notebook only **calls the repository scripts**; all logic lives in `src/m2cl/` so Colab, a laptop and the paper use identical code.

| Step | What | T4 time (approx.) |
|---|---|---|
| 0 | Setup, smoke test | 5 min |
| 1 | Corrected M2-CL, 6 taps, 4 targets × 3 seeds | ~3 h |
| 2 | Matched ERM, 4 × 3 | ~2 h |
| 3 | **Missing ablation**: 6-tap M2 without contrastive loss, 4 × 3 | ~3 h |
| 4 | Feature extraction for leakage probes | ~20 min |
| 5 | Leakage probes (raw + z-scored) | 3–5 h per run (probes use the CPU) |
| 6 | All tables | 1 min |

Every run checkpoints each epoch to Google Drive and is skipped once complete, so you can **re-run any cell after a disconnect** and it continues where it stopped.

In [ ]:
# ── 0. Setup ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

REPO_ON_DRIVE = '/content/drive/MyDrive/M2CL-Reproduction'   # upload the repo folder here
PACS_ZIP      = '/content/drive/MyDrive/PACS.zip'             # your PACS zip
OUT           = '/content/drive/MyDrive/m2cl_v2'              # all results go here (persistent)

import os, shutil, subprocess, zipfile
shutil.rmtree('/content/repo', ignore_errors=True)
shutil.copytree(REPO_ON_DRIVE, '/content/repo')
%cd /content/repo
!pip -q install -r requirements.txt

if not os.path.exists('/content/PACS_raw'):
    zipfile.ZipFile(PACS_ZIP).extractall('/content/PACS_raw')
# locate the RGB copy: the folder with the four domains whose files are not dct_* (audit A0)
def _is_rgb(root):
    folder = os.path.join(root, 'art_painting', 'dog')
    return os.path.isdir(folder) and not any(f.startswith('dct_') for f in os.listdir(folder))
DATA = next(r for r, d, _ in os.walk('/content/PACS_raw')
            if {'art_painting', 'cartoon', 'photo', 'sketch'} <= set(d) and _is_rgb(r))
print('PACS root:', DATA)
os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Optional but recommended once: whole pipeline on a tiny fake dataset (~3 min)
!python scripts/smoke_test.py

## Fast path: re-run the draft's leakage tables on real images (no training, about 1-1.5 h)
The draft's Tables II, III and IX were probed on the DCT copy of PACS (docs/AUDIT.md, A0). Upload `m2cl_checkpoints/` and `m2cl6tap/` to `MyDrive/legacy_ckpts/`, run the setup cell, then this cell. It probes only the 512-d backbone feature; per-tap probes come later.

In [ ]:
!python scripts/extract_features.py --manifest configs/leakage_manifest.yaml     --checkpoint_root /content/drive/MyDrive/legacy_ckpts --data_root "$DATA" --out_dir "$OUT/features"
!python scripts/run_leakage.py --features_dir "$OUT/features" --out "$OUT/leakage/raw_backbone" --layers backbone
!python scripts/make_tables.py --leakage "$OUT/leakage/raw_backbone" --comparisons configs/comparisons.yaml --out "$OUT/tables/raw_backbone"
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/tables/raw_backbone/tables.md').read()))

## 1–3. Training (each cell is resumable)

In [ ]:
!python scripts/train_lodo.py --config configs/m2cl_6tap.yaml --data_root "$DATA" --out_dir "$OUT/runs" 

In [ ]:
!python scripts/train_lodo.py --config configs/erm.yaml --data_root "$DATA" --out_dir "$OUT/runs" 

In [ ]:
!python scripts/train_lodo.py --config configs/m2_6tap.yaml --data_root "$DATA" --out_dir "$OUT/runs" 

## 4. Feature extraction
New runs are picked up automatically (`--from_runs`). To also re-probe the **original checkpoints** used in the draft, upload `m2cl_checkpoints/` and `m2cl6tap/` to `MyDrive/legacy_ckpts/` (the manifest skips anything missing).

In [ ]:
!python scripts/extract_features.py --manifest configs/leakage_manifest.yaml \
    --checkpoint_root /content/drive/MyDrive/legacy_ckpts --from_runs "$OUT/runs" \
    --data_root "$DATA" --out_dir "$OUT/features" 

## 5. Leakage probes (5 probe seeds; raw features = draft protocol, z-scored = robustness check)

In [ ]:
!python scripts/run_leakage.py --features_dir "$OUT/features" --out "$OUT/leakage/raw"
!python scripts/run_leakage.py --features_dir "$OUT/features" --out "$OUT/leakage/zscore" --standardize

## 6. Tables

In [ ]:
!python scripts/make_tables.py --runs "$OUT/runs" --acc_compare m2cl_6tap:erm m2cl_6tap:m2_6tap m2_6tap:erm \
    --leakage "$OUT/leakage/raw" --comparisons configs/comparisons.yaml --out "$OUT/tables/raw"
!python scripts/make_tables.py --leakage "$OUT/leakage/zscore" --comparisons configs/comparisons.yaml --out "$OUT/tables/zscore"
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/tables/raw/tables.md').read()))